# 00. 데이터 수집·정제

2024년과 2025년 MLB 정규시즌의 투구 데이터를 내려받아, 심판이 볼과 스트라이크를
직접 판정한 공만 골라낸다. 여기에 분석에 쓸 변수를 붙여 `data/clean.parquet`
하나로 저장한다. 뒤에 이어지는 노트북은 모두 이 파일을 읽는 데서 시작한다.

2026년에는 ABS(자동 볼·스트라이크 판정 시스템)가 도입되면서 존을 정하는 기준이
바뀌었다. 그래서 그 이전의 두 시즌만 쓴다.

## 사용 도구

| 도구 | 쓰는 곳 |
|---|---|
| [pybaseball](https://github.com/jldbc/pybaseball) | MLB 투구 데이터(Statcast)를 파이썬으로 내려받는다 |
| [pandas](https://pandas.pydata.org/) | 표 형태의 데이터를 거르고, 합치고, 집계한다 |
| [NumPy](https://numpy.org/) | 거리 계산 같은 수치 연산 |
| [pyarrow](https://arrow.apache.org/docs/python/) | 데이터를 [parquet](https://parquet.apache.org/) 파일로 저장한다. parquet은 열 단위로 압축하는 파일 형식이라 CSV보다 훨씬 작고, 날짜나 정수 같은 자료형이 그대로 남는다 |
| [Matplotlib](https://matplotlib.org/), [seaborn](https://seaborn.pydata.org/) | 그래프 (다음 노트북부터 쓴다) |

필요한 패키지는 저장소 루트에서 `pip install -r requirements.txt`로 한 번에
설치할 수 있다.

## 상수

분석 전체에서 쓰는 상수를 맨 앞에 모아 둔다. 길이는 모두 피트 단위다(1ft ≈ 30.5cm).

| 상수 | 값 | 뜻 |
|---|---|---|
| `PLATE_HALF` | 0.708 | 홈플레이트 폭(17인치)의 절반. 스트라이크 존의 좌우 경계다 |
| `BALL_RADIUS` | 0.121 | 야구공 반지름(약 1.45인치) |
| `BORDERLINE_THRESHOLD` | 0.121 | 존 경계에서 이 거리 안쪽으로 들어온 공을 경계 공으로 본다. 이유는 4절에 있다 |
| `MIN_CELL_N` | 200 | 조건별로 나눠 비교할 때 한 칸에 적어도 이만큼은 있어야 한다. 더 적으면 신뢰구간이 너무 넓어 결론을 내기 어렵다 |
| `SPLIT_DATE` | 2025-08-01 | 모델을 만들 때 이 날짜 전 데이터로 학습하고, 이후 데이터로 평가한다 |

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("../data")
MODEL_DIR = Path("../models")

PLATE_HALF = 0.708            # 홈플레이트 반폭 (ft)
BALL_RADIUS = 0.121           # 공 반지름 (ft)
BORDERLINE_THRESHOLD = 0.121  # 경계 공 기준 (ft)
MIN_CELL_N = 200              # 셀당 최소 표본 수
SPLIT_DATE = "2025-08-01"     # 이 날짜 전은 모델 학습용, 이후는 평가용

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

아래는 이 노트북에서만 쓰는 값이다.

`SEASONS`는 두 시즌의 개막일과 정규시즌 마지막 날이다. 2024년은 서울 시리즈
(3월 20일), 2025년은 도쿄 시리즈(3월 18일)로 시즌을 시작했다. 포스트시즌은 이
기간 밖이라 받지 않고, 혹시 섞여 들어오더라도 뒤에서 `game_type`으로 한 번 더 거른다.

`CALLED`는 판정구에 해당하는 투구 결과 세 가지다. 어떤 결과들이 있고 왜 이 셋만
남기는지는 3절에서 표와 함께 설명한다. `SZ_TOP_MAX`는 존 상단 높이의 상한으로,
이보다 높게 기록된 공은 측정 오류로 보고 뺀다.

`KEEP_COLS`는 원본 컬럼 119개 가운데 실제로 쓰는 30개이고, 뜻은 2절에 정리했다.
`FRAMING_URL`은 5절에서 포수 프레이밍 점수를 받아 오는 주소다.

In [2]:
import contextlib
import io
import urllib.request
import warnings

from pybaseball import statcast

RAW_DIR = DATA_DIR / "raw"

SEASONS = {
    2024: ("2024-03-20", "2024-09-30"),
    2025: ("2025-03-18", "2025-09-28"),
}

CALLED = ["called_strike", "ball", "blocked_ball"]
SZ_TOP_MAX = 5.0  # 존 상단이 이보다 높으면 측정 오류로 본다 (ft)

KEEP_COLS = [
    "game_date", "game_pk", "game_year", "game_type",  # 날짜, 경기 ID, 시즌, 경기 유형(R = 정규시즌)
    "at_bat_number", "pitch_number",                   # 경기 안 타석 번호, 타석 안 투구 번호
    "description",                                     # 투구 결과 (판정 포함)
    "plate_x", "plate_z",                              # 홈플레이트 통과 위치 (좌우, 높이)
    "sz_top", "sz_bot",                                # 그 타자의 스트라이크 존 상단·하단 높이
    "balls", "strikes",                                # 투구 전 볼카운트
    "outs_when_up", "inning", "inning_topbot",         # 아웃 수, 이닝, 초(Top)/말(Bot)
    "on_1b", "on_2b", "on_3b",                         # 1·2·3루 주자 (주자 선수 ID, 없으면 결측)
    "bat_score", "fld_score",                          # 공격 팀 점수, 수비 팀 점수
    "pitcher", "batter", "fielder_2",                  # 투수·타자·포수의 MLB 선수 ID
    "stand", "p_throws",                               # 타자가 선 쪽(L/R), 투수가 던지는 손(L/R)
    "pitch_type", "release_speed",                     # 구종, 구속 (mph)
    "home_team", "away_team",                          # 홈팀, 원정팀
]

FRAMING_URL = (
    "https://baseballsavant.mlb.com/leaderboard/catcher-framing"
    "?type=catcher&seasonStart={year}&seasonEnd={year}&team=&min=q"
    "&sortColumn=rv_tot&sortDirection=desc&csv=true"
)

## 1. 수집

데이터는 MLB의 투구 추적 시스템 [Statcast](https://baseballsavant.mlb.com/statcast_search)에서
[pybaseball](https://github.com/jldbc/pybaseball)로 받는다. Statcast에는 모든 투구에
대해 홈플레이트를 지난 위치(`plate_x`, `plate_z`), 그 타자의 스트라이크 존 높이
(`sz_top`, `sz_bot`), 심판의 판정(`description`)이 남아 있다.

두 시즌을 통째로 요청하지 않고 한 달씩 나눠 받는다. 한 번에 받을 수 있는 행 수에
상한이 있어서, 기간을 길게 잡으면 데이터 일부가 경고 없이 잘려 나갈 수 있다.
다운로드도 오래 걸려 도중에 끊기기 쉽다.

받은 달은 그때그때 `data/raw/`에 저장한다. 아래 셀은 저장된 파일이 있으면 다시
받지 않고 그 파일을 읽으므로, 처음 실행할 때만 몇 분 걸리고 다음부터는 금방
끝난다. 중간에 끊겨도 다시 실행하면 못 받은 달부터 이어서 받는다. 원본 파일은 다
합쳐 약 230MB라 저장소에는 올리지 않았다.

In [3]:
def month_ranges(start, end):
    """[start, end] 기간을 달력 월 단위 (시작일, 종료일) 목록으로 자른다."""
    cur, end = pd.Timestamp(start), pd.Timestamp(end)
    ranges = []
    while cur <= end:
        month_end = min(cur + pd.offsets.MonthEnd(0), end)
        ranges.append((cur, month_end))
        cur = month_end + pd.Timedelta(days=1)
    return ranges


def fetch_month(start, end):
    """한 달 치 원본을 받아 저장한다. 이미 있으면 받지 않는다."""
    path = RAW_DIR / f"statcast_{start:%Y_%m}.parquet"
    if path.exists():
        return path, "기존 파일"
    # pybaseball의 진행 막대와 경고는 노트북 출력만 어지럽혀 숨긴다
    with warnings.catch_warnings(), contextlib.redirect_stderr(io.StringIO()):
        warnings.simplefilter("ignore")
        df = statcast(start_dt=f"{start:%Y-%m-%d}", end_dt=f"{end:%Y-%m-%d}", verbose=False)
    df.to_parquet(path, index=False)
    return path, "새로 받음"


RAW_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for year, (start, end) in SEASONS.items():
    for s, e in month_ranges(start, end):
        path, status = fetch_month(s, e)
        ids = pd.read_parquet(path, columns=["game_pk"])["game_pk"]
        rows.append({
            "시즌": year,
            "파일": path.name,
            "기간": f"{s:%m-%d} ~ {e:%m-%d}",
            "투구 수": len(ids),
            "경기 수": ids.nunique(),
            "경기당 투구": round(len(ids) / ids.nunique()),
            "상태": status,
        })

raw_summary = pd.DataFrame(rows)
raw_summary

,시즌,파일,기간,투구 수,경기 수,경기당 투구,상태
0,2024,statcast_2024_03.parquet,03-20 ~ 03-31,37217,143,260,기존 파일
1,2024,statcast_2024_04.parquet,04-01 ~ 04-30,116769,397,294,기존 파일
2,2024,statcast_2024_05.parquet,05-01 ~ 05-31,118739,409,290,기존 파일
3,2024,statcast_2024_06.parquet,06-01 ~ 06-30,116355,401,290,기존 파일
4,2024,statcast_2024_07.parquet,07-01 ~ 07-31,108693,369,295,기존 파일
5,2024,statcast_2024_08.parquet,08-01 ~ 08-31,121441,413,294,기존 파일
6,2024,statcast_2024_09.parquet,09-01 ~ 09-30,113268,385,294,기존 파일
7,2025,statcast_2025_03.parquet,03-18 ~ 03-31,49338,168,294,기존 파일
8,2025,statcast_2025_04.parquet,04-01 ~ 04-30,114767,391,294,기존 파일
9,2025,statcast_2025_05.parquet,05-01 ~ 05-31,120218,411,293,기존 파일


월별 투구 수는 10만~12만 구 사이이고, 경기당 290구 안팎으로 고르게 나온다.
데이터가 빠진 것처럼 보이는 달은 없다. 3월은 개막 후 며칠 치라 투구 수가 적은데,
시범경기가 섞여 있어서 경기 수는 실제보다 많게 잡혔다. 시범경기는 3절에서
빠진다. 7월에 경기가 조금 적은 건 올스타 휴식기 때문이다.

In [4]:
raw_summary.groupby("시즌")[["투구 수", "경기 수"]].sum()

,투구 수,경기 수
시즌,,
2024,732482,2517
2025,742080,2531


두 시즌을 합치면 약 147만 구다. 여기에는 시범경기가 들어 있고, 정규시즌만 남긴
수는 3절의 정제 표에서 확인할 수 있다.

## 2. 필요한 컬럼만 불러오기

원본에는 컬럼이 119개나 있지만 타구 속도, 회전수, 수비 위치처럼 판정과 관계없는
정보가 대부분이다. 그래서 아래 30개만 읽는다. 각 컬럼의 공식 정의는
[Baseball Savant CSV 문서](https://baseballsavant.mlb.com/csv-docs)에서 볼 수 있다.

| 용도 | 컬럼 | 뜻 |
|---|---|---|
| 경기·투구 식별 | `game_date`, `game_pk`, `game_year` | 경기 날짜, 경기 고유 ID, 시즌 |
| | `game_type` | 경기 유형. `R`이 정규시즌, `S`가 시범경기 |
| | `at_bat_number`, `pitch_number` | 경기 안에서 몇 번째 타석인지, 그 타석에서 몇 번째 공인지 |
| 판정 | `description` | 투구 결과. 볼·스트라이크 판정, 스윙, 파울, 인플레이 등 |
| 투구 위치 | `plate_x` | 공이 홈플레이트를 지날 때의 좌우 위치(ft). 0이 플레이트 중앙, 포수 시점에서 오른쪽이 양수 |
| | `plate_z` | 같은 지점에서의 높이(ft, 지면 기준) |
| | `sz_top`, `sz_bot` | 그 타자의 스트라이크 존 상단·하단 높이(ft). 타자 키와 자세에 따라 다르다 |
| 볼카운트 | `balls`, `strikes` | 그 공을 던지기 전의 볼, 스트라이크 개수 |
| 경기 상황 | `outs_when_up` | 그 공을 던지기 전의 아웃 수 |
| | `inning`, `inning_topbot` | 이닝, 초(`Top`, 원정팀 공격)/말(`Bot`, 홈팀 공격) |
| | `on_1b`, `on_2b`, `on_3b` | 각 베이스의 주자 선수 ID. 주자가 없으면 비어 있다 |
| | `bat_score`, `fld_score` | 공격 팀, 수비 팀 점수 |
| 선수 | `pitcher`, `batter`, `fielder_2` | 투수, 타자, 포수의 MLB 선수 ID |
| | `stand`, `p_throws` | 타자가 선 타석(`L`/`R`), 투수가 던지는 손(`L`/`R`) |
| | `pitch_type`, `release_speed` | 구종(예: `FF` 포심 패스트볼), 구속(mph) |
| | `home_team`, `away_team` | 홈팀, 원정팀 약칭 |

In [5]:
raw_files = sorted(RAW_DIR.glob("statcast_*.parquet"))
raw = pd.concat(
    (pd.read_parquet(p, columns=KEEP_COLS) for p in raw_files),
    ignore_index=True,
)
print(f"{len(raw):,}행 × {raw.shape[1]}열")
raw.head()

1,474,562행 × 30열


,game_date,game_pk,game_year,game_type,at_bat_number,pitch_number,description,plate_x,plate_z,sz_top,...,fld_score,pitcher,batter,fielder_2,stand,p_throws,pitch_type,release_speed,home_team,away_team
0,2024-03-31,747222,2024,R,66,7,swinging_strike,0.077266,3.227396,3.42,...,5,656464,686668,571466,R,R,FF,95.7,AZ,COL
1,2024-03-31,747222,2024,R,66,6,foul,-0.044209,1.73542,3.42,...,5,656464,686668,571466,R,R,SI,94.6,AZ,COL
2,2024-03-31,747222,2024,R,66,5,blocked_ball,1.722746,0.268207,3.41638,...,5,656464,686668,571466,R,R,SL,88.4,AZ,COL
3,2024-03-31,747222,2024,R,66,4,foul,0.270552,1.738154,3.42,...,5,656464,686668,571466,R,R,SL,86.9,AZ,COL
4,2024-03-31,747222,2024,R,66,3,ball,0.162886,1.022717,3.339202,...,5,656464,686668,571466,R,R,SI,94.7,AZ,COL


## 3. 정제

이제 분석에 쓰지 않을 공을 하나씩 걸러 낸다. 나중에 같은 결과를 다시 만들 수
있도록, 단계마다 몇 개가 남고 몇 개가 빠졌는지 표로 기록해 둔다.

첫 단계로 정규시즌 경기만 남긴다. 같은 투구가 두 번 들어온 행(경기, 타석, 투구
번호가 모두 같은 행)이 있는지도 함께 확인한다.

In [6]:
clean_log = []


def record(step, df):
    prev = clean_log[-1]["남은 행"] if clean_log else len(df)
    clean_log.append({"단계": step, "남은 행": len(df), "제거된 행": prev - len(df)})


record("0. 원본 (2024-2025 전체 투구)", raw)

regular = raw[raw["game_type"] == "R"]
record("1. 정규시즌만", regular)

regular = regular.drop_duplicates(["game_pk", "at_bat_number", "pitch_number"])
record("1-1. 중복 투구 제거", regular)

### 어떤 공이 판정구인가

이 분석에서 보는 공은 타자가 치지 않아서 심판이 존 안인지 밖인지 판단해야 했던
공, 즉 판정구다. 정규시즌 투구의 결과(`description`)가 어떻게 나뉘는지부터 본다.

In [7]:
outcomes = regular["description"].value_counts().rename("투구 수").to_frame()
outcomes["비율(%)"] = (outcomes["투구 수"] / len(regular) * 100).round(1)
outcomes["판정구"] = np.where(outcomes.index.isin(CALLED), "O", "")
outcomes

,투구 수,비율(%),판정구
description,,,
ball,473998,33.3,O
foul,256078,18.0,
hit_into_play,249106,17.5,
called_strike,231782,16.3,O
swinging_strike,148888,10.5,
blocked_ball,30162,2.1,O
foul_tip,14655,1.0,
swinging_strike_blocked,7932,0.6,
automatic_ball,4597,0.3,


투구 결과는 15가지로 나뉜다. 이 가운데 심판이 존을 보고 판단한 것은 `ball`(볼),
`called_strike`(스트라이크 판정), `blocked_ball`(원바운드 등으로 포수가 몸으로 막아
낸 볼) 세 가지이고, 합치면 정규시즌 투구의 51.7%다.

나머지는 심판의 존 판단과 상관이 없어서 뺀다. 파울이나 인플레이 타구처럼 타자가
공을 맞힌 경우(`foul`, `hit_into_play`, `foul_tip`, `foul_bunt`, `bunt_foul_tip`)는
이미 스윙을 했으니 판정할 일이 없다. 헛스윙(`swinging_strike`,
`swinging_strike_blocked`, `missed_bunt`)은 위치와 상관없이 스트라이크다.
`automatic_ball`과 `automatic_strike`는 피치 클락 위반 같은 이유로 공을 보지 않고
주어지는 판정이다. 몸에 맞는 공(`hit_by_pitch`)이면 타자는 그냥 1루로 나가고,
피치아웃(`pitchout`)은 도루를 막으려고 일부러 존 밖으로 크게 빼는 공이라 판정이
갈릴 일이 없다.

### 판정구, 결측, 이상치

위 표에서 O로 표시한 세 가지만 남긴 뒤, 추적 장비가 공을 놓쳐 위치가 비어 있는
행을 지운다. 그리고 존 높이가 이상하게 기록된 공이 있는지 본다. 존 상단이 하단보다
낮거나 5ft(약 152cm)를 넘으면 측정 오류로 보고 뺀다.

In [8]:
called = regular[regular["description"].isin(CALLED)]
record("2. 판정구만", called)

called = called.dropna(subset=["plate_x", "plate_z", "sz_top", "sz_bot"])
record("3. 위치 결측 제거", called)

called[["sz_top", "sz_bot"]].describe().round(2)

,sz_top,sz_bot
count,735718.0,735718.0
mean,3.42,1.6
std,0.2,0.12
min,2.51,1.0
25%,3.28,1.52
50%,3.42,1.6
75%,3.55,1.68
max,4.41,2.14


존 상단은 2.5~4.4ft, 하단은 1.0~2.1ft 사이로 모두 사람 키로 납득할 만한 값이다.
상단이 하단보다 낮거나 5ft를 넘는 경우는 없다.

In [9]:
bad_zone = (called["sz_top"] <= called["sz_bot"]) | (called["sz_top"] > SZ_TOP_MAX)
called = called[~bad_zone]
record("4. 존 이상치 제거", called)

pd.DataFrame(clean_log)

,단계,남은 행,제거된 행
0,0. 원본 (2024-2025 전체 투구),1474562,0
1,1. 정규시즌만,1424427,50135
2,1-1. 중복 투구 제거,1424427,0
3,2. 판정구만,735942,688485
4,3. 위치 결측 제거,735718,224
5,4. 존 이상치 제거,735718,0


정규시즌 필터에서 빠진 5만 구 남짓은 3월 시범경기다. 같은 투구가 두 번 들어온
경우는 없었다. 정규시즌 투구 가운데 판정구는 절반을 조금 넘고(51.7%), 나머지는
스윙이나 파울, 인플레이처럼 심판이 존을 판단하지 않은 공이다. 위치가 비어 있던
공은 224개(판정구의 0.03%)뿐이었고 존 높이 이상치는 없었다.

이렇게 해서 735,718구가 남았다. 정규시즌 4,859경기(2024년 2,429경기, 2025년
2,430경기)에서 던진 공이다.

## 4. 파생 변수

정제한 데이터에 분석용 변수를 붙인다. 먼저 판정과 투구 위치에 관한 변수다.

스트라이크 존은 홈플레이트 폭(17인치)과 타자의 무릎에서 가슴까지의 높이로
만들어지는 가상의 상자다. 위아래 경계는 타자의 키와 자세에 따라 달라서, Statcast가
타자마다 `sz_top`, `sz_bot`으로 따로 기록한다. 규칙상 공이 이 상자에 조금이라도
걸치면 스트라이크다.

- `is_strike`: 스트라이크 판정이면 1, 볼이면 0이다.
- `zone_dist`: 공의 중심에서 존 경계까지의 거리(ft)로, 존 안쪽이면 양수, 바깥쪽이면
  음수다. 공이 걸치기만 해도 스트라이크이므로 존을 상하좌우로 공 반지름만큼 넓혀
  놓고 잰다. 좌우 경계와 상하 경계 중 더 가까운 쪽까지의 거리를 쓴다.
- `plate_z_norm`: 존 하단을 0, 상단을 1로 놓았을 때의 투구 높이다. 키가 다른
  타자들의 공을 같은 잣대로 비교하려고 만든다.
- `is_borderline`: `zone_dist`의 절댓값이 0.121ft 이하인 공, 즉 경계 공이다. 존
  경계선을 가운데 두고 공 하나 폭(약 2.9인치)만큼의 띠에 들어온 공이다.

경계 공을 따로 표시하는 데는 이유가 있다. 한가운데로 들어온 공은 어떤 상황에서든
스트라이크이고 한참 빠진 공은 어떤 상황에서든 볼이라, 상황이나 선수에 따라 판정이
흔들릴 수 있는 건 경계 근처의 공뿐이다. 게다가 투수는 볼카운트에 따라 던지는 곳을
바꾼다. 3-0에서는 가운데로 넣고, 0-2에서는 존 밖으로 유인한다. 그래서 전체 공을
놓고 비교하면 심판이 아니라 투수의 선택을 비교하게 된다. 경계 공끼리 비교하면 양쪽
다 아슬아슬한 공이라 이런 차이가 줄어든다. 띠의 폭을 공 하나로 잡은 건 "공이
걸치면 스트라이크"라는 규칙 자체가 공의 크기를 기준으로 하기 때문이다.

In [10]:
clean = called.copy()

for col in ["plate_x", "plate_z", "sz_top", "sz_bot", "release_speed"]:
    clean[col] = clean[col].astype("float64")

clean["is_strike"] = (clean["description"] == "called_strike").astype("int8")
clean["plate_z_norm"] = (clean["plate_z"] - clean["sz_bot"]) / (clean["sz_top"] - clean["sz_bot"])

dx = (PLATE_HALF + BALL_RADIUS) - clean["plate_x"].abs()
dz = np.minimum(
    (clean["sz_top"] + BALL_RADIUS) - clean["plate_z"],
    clean["plate_z"] - (clean["sz_bot"] - BALL_RADIUS),
)
clean["zone_dist"] = np.minimum(dx, dz)
clean["is_borderline"] = (clean["zone_dist"].abs() <= BORDERLINE_THRESHOLD).astype("int8")

다음은 경기 상황 변수다. 볼카운트는 공을 던지기 전의 볼과 스트라이크 개수로,
3-0이라고 하면 볼 3개에 스트라이크 0개라는 뜻이다. 원본의 `balls`, `strikes`를
그대로 쓰고, 나머지는 새로 만든다.

- `score_diff`: 공격 팀 점수에서 수비 팀 점수를 뺀 값
- `risp`: 2루나 3루에 주자가 있으면 1. 안타 하나면 점수가 날 수 있는 득점권 상황이다
- `late_close`: 7회 이후이면서 점수 차가 2점 이내이면 1. 세이버메트릭스에서 쓰는
  Late & Close(경기 후반 접전)를 단순하게 옮겼다
- `is_home_batting`: 홈팀이 공격 중(말 공격)이면 1

In [11]:
clean["score_diff"] = (clean["bat_score"] - clean["fld_score"]).astype("int16")
clean["risp"] = (clean["on_2b"].notna() | clean["on_3b"].notna()).astype("int8")
clean["late_close"] = ((clean["inning"] >= 7) & (clean["score_diff"].abs() <= 2)).astype("int8")
clean["is_home_batting"] = (clean["inning_topbot"] == "Bot").astype("int8")

clean[["is_strike", "is_borderline", "risp", "late_close", "is_home_batting"]].mean().round(3)

is_strike          0.315
is_borderline      0.163
risp               0.230
late_close         0.164
is_home_batting    0.490
dtype: float64

판정구 가운데 31.5%가 스트라이크다. 경계 공은 16.3%로 약 12만 개인데, 상황이나
선수별로 쪼개더라도 조건마다 수천 개씩은 남을 만한 양이다. 조건별로 실제 몇 개씩
남는지는 `01_baseline`에서 교차표로 따져 본다.

득점권 상황은 23.0%, 후반 접전은 16.4%다. 홈팀 공격 비율이 50%에 조금 못 미치는
건 홈팀이 이기고 있으면 9회 말을 하지 않기 때문이다.

## 5. 포수 프레이밍 점수

프레이밍은 포수가 공을 받는 동작으로 경계 공을 스트라이크처럼 보이게 만드는
기술이다. 같은 위치의 공이라도 누가 받느냐에 따라 판정이 달라질 수 있어서, 포수의
영향을 따로 떼어 내려면 이 점수가 필요하다.

점수는 [Baseball Savant의 Catcher Framing 리더보드](https://baseballsavant.mlb.com/leaderboard/catcher-framing)에서
시즌별 CSV로 받아 `data/framing_YYYY.csv`에 저장한다. 파일이 작아서 저장소에 함께
올려 두었다.

리더보드의 `rv_tot`는 프레이밍으로 얻은 시즌 누적 득점이다. 누적값이라 많이 출장한
포수일수록 커지기 때문에, 받은 판정구 수(`pitches`)로 나눠 판정구 1,000개당 득점으로
바꾼다. 이렇게 하면 출장량과 상관없이 포수의 실력을 비교할 수 있다. 이 값을
`framing_score`라 부르고, 0이 리그 평균이다. 투구 데이터와는 포수의 MLB 선수
ID(`fielder_2`와 리더보드의 `id`)로 연결하고, 2024년 점수는 2024년 투구에만,
2025년 점수는 2025년 투구에만 붙인다.

리더보드에는 출장 기준을 채운 주전급 포수만 올라 있다. 기준에 못 미친 백업 포수는
점수가 없어서 리그 평균인 0으로 채우고, 그런 공이 얼마나 되는지 기록해 둔다.

In [12]:
def load_framing(year):
    path = DATA_DIR / f"framing_{year}.csv"
    if not path.exists():
        req = urllib.request.Request(FRAMING_URL.format(year=year), headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp:
            path.write_bytes(resp.read())
    return pd.read_csv(path, encoding="utf-8-sig").assign(game_year=year)


framing = pd.concat([load_framing(y) for y in SEASONS], ignore_index=True)
framing["framing_score"] = framing["rv_tot"] / framing["pitches"] * 1000

framing.groupby("game_year")["framing_score"].describe().round(2)

,count,mean,std,min,25%,50%,75%,max
game_year,,,,,,,,
2024,58.0,-0.01,1.10,-2.10,-0.88,0.03,0.72,2.76
2025,57.0,-0.06,1.02,-2.59,-0.73,-0.09,0.49,2.81


리더보드에는 시즌마다 57~58명이 올라 있다. `framing_score`는 대략 -2.6에서 +2.8
사이로, 판정구 1,000개당 많게는 3점 가까이 차이가 난다. 평균이 0에 가까워서 점수가
없는 포수를 0으로 채워도 크게 무리가 없다.

In [13]:
framing_key = framing[["game_year", "id", "framing_score"]].rename(columns={"id": "fielder_2"})
for col in ["game_year", "fielder_2"]:
    clean[col] = clean[col].astype("int64")
    framing_key[col] = framing_key[col].astype("int64")

clean = clean.merge(framing_key, on=["game_year", "fielder_2"], how="left")

framing_missing = clean.groupby("game_year")["framing_score"].apply(lambda s: s.isna().mean())
print("프레이밍 점수가 없어 0으로 채운 판정구 비율")
print((framing_missing * 100).round(1).astype(str) + "%")

clean["framing_score"] = clean["framing_score"].fillna(0.0)

프레이밍 점수가 없어 0으로 채운 판정구 비율
game_year
2024    10.8%
2025    12.7%
Name: framing_score, dtype: object


점수가 없어 0으로 채운 공은 2024년 10.8%, 2025년 12.7%다. 이 공들은 포수의
실력을 리그 평균으로 가정했기 때문에 포수 효과가 그만큼 덜 걸러진다. 결론에서
한계로 짚는다.

점수를 판정구 1,000개당이라고 부르려면 리더보드의 `pitches`가 정말 포수가 받은
판정구 수여야 한다. 우리 데이터에서 직접 센 값과 맞춰 본다.

In [14]:
ours = clean.groupby(["game_year", "fielder_2"]).size().rename("우리 판정구 수")
board = framing.set_index(["game_year", "id"]).rename_axis(["game_year", "fielder_2"])
compare = board[["name", "pitches", "rv_tot", "pct_tot"]].join(ours)

print(f"pitches / 우리 판정구 수 (중앙값): {(compare['pitches'] / compare['우리 판정구 수']).median():.3f}")
print(f"리더보드 최소 pitches: {compare['pitches'].min():,}")
compare.groupby("game_year").agg(포수=("name", "size"), 섀도존_스트라이크율=("pct_tot", "mean")).round(3)

pitches / 우리 판정구 수 (중앙값): 1.000
리더보드 최소 pitches: 2,549


,포수,섀도존_스트라이크율
game_year,,
2024,58,0.465
2025,57,0.420


`pitches`는 우리가 센 포수별 판정구 수와 정확히 같다(비율 1.000). 따라서
`framing_score`는 판정구 1,000개당 프레이밍 득점이 맞다. 리더보드에 오르는 기준은
판정구 약 2,550개다.

`pct_tot`는 존 경계 부근(섀도 존)에서 스트라이크가 나온 비율인데, 2024년 46.5%에서
2025년 42.0%로 떨어졌다. 뒤에서 살펴볼 2025년 판정 존의 변화가 Savant의 집계에서도
같은 방향으로 나타난다.

## 6. 저장

컬럼을 보기 좋은 순서로 정리하고, 용량을 줄이려고 자료형을 바꾼다. 선수·경기 ID는
정수(`int32`)로, 0/1 변수는 `int8`로 둔다. 주자 컬럼(`on_1b` 등)에는 주자의 선수
ID가 들어 있고 베이스가 비면 값이 없어서, 빈 값을 허용하는 정수형(`Int32`)으로 둔다.

In [15]:
ID_COLS = ["game_pk", "game_year", "at_bat_number", "pitch_number", "pitcher", "batter", "fielder_2",
           "balls", "strikes", "outs_when_up", "inning", "bat_score", "fld_score"]
for col in ID_COLS:
    clean[col] = clean[col].astype("int32")
for col in ["on_1b", "on_2b", "on_3b"]:
    clean[col] = clean[col].astype("Int32")

ORDER = [
    # 식별
    "game_date", "game_pk", "game_year", "game_type", "at_bat_number", "pitch_number",
    # 판정
    "description", "is_strike",
    # 위치
    "plate_x", "plate_z", "sz_top", "sz_bot", "plate_z_norm", "zone_dist", "is_borderline",
    # 상황
    "balls", "strikes", "outs_when_up", "inning", "inning_topbot",
    "on_1b", "on_2b", "on_3b", "bat_score", "fld_score",
    "score_diff", "risp", "late_close", "is_home_batting",
    # 선수
    "pitcher", "batter", "fielder_2", "stand", "p_throws",
    "pitch_type", "release_speed", "home_team", "away_team",
    # 통제
    "framing_score",
]
assert sorted(ORDER) == sorted(clean.columns), set(ORDER) ^ set(clean.columns)
clean = clean[ORDER].sort_values(["game_date", "game_pk", "at_bat_number", "pitch_number"], ignore_index=True)

out_path = DATA_DIR / "clean.parquet"
clean.to_parquet(out_path, index=False)
print(f"{out_path}: {len(clean):,}행 × {clean.shape[1]}열, {out_path.stat().st_size / 1e6:.1f} MB")

../data/clean.parquet: 735,718행 × 39열, 43.9 MB


735,718행, 39열에 약 44MB다. GitHub의 파일 크기 제한(100MB)보다 작아서 저장소에
함께 올렸다. 다음 노트북부터는 데이터를 다시 받을 필요 없이 이 파일만 읽으면 된다.

저장한 파일을 다시 읽어서 의도대로 만들어졌는지 확인한다. 판정구만 들어 있는지,
위치와 프레이밍 점수에 빈 값이 없는지, 같은 투구가 두 번 들어가지 않았는지 검사하고
시즌별 요약을 낸다.

In [16]:
check = pd.read_parquet(out_path)

assert check["description"].isin(CALLED).all()
assert check[["plate_x", "plate_z", "sz_top", "sz_bot", "framing_score"]].notna().all().all()
assert not check.duplicated(["game_pk", "at_bat_number", "pitch_number"]).any()
assert (check["is_strike"] == (check["description"] == "called_strike")).all()

summary = check.groupby("game_year").agg(
    판정구=("is_strike", "size"),
    스트라이크_비율=("is_strike", "mean"),
    경계공=("is_borderline", "sum"),
    경계공_스트라이크_비율=("is_strike", lambda s: s[check.loc[s.index, "is_borderline"] == 1].mean()),
)
summary.round(3)

,판정구,스트라이크_비율,경계공,경계공_스트라이크_비율
game_year,,,,
2024,366873,0.317,59505,0.502
2025,368845,0.313,60548,0.427


검사는 모두 통과했다. 그런데 시즌별 요약에서 눈에 띄는 차이가 하나 있다. 전체
스트라이크 비율은 2024년 31.7%, 2025년 31.3%로 거의 같은데, 경계 공만 보면 50.2%에서
42.7%로 7.5%p나 떨어진다. 이 차이는 데이터를 한 번 훑어본 뒤 따로 살펴본다.

정제를 마친 데이터를 직접 확인한다. 컬럼이 39개라 옆으로 너무 길어서, 경계 공 세
개를 행과 열을 바꿔 세로로 늘어놓았다.

In [17]:
check[check["is_borderline"] == 1].head(3).T

,1,8,18
game_date,2024-03-20 00:00:00,2024-03-20 00:00:00,2024-03-20 00:00:00
game_pk,745444,745444,745444
game_year,2024,2024,2024
game_type,R,R,R
at_bat_number,1,4,8
pitch_number,3,1,1
description,ball,ball,called_strike
is_strike,0,0,1
plate_x,0.727241,0.94543,-0.720917
plate_z,2.519773,1.631631,2.749523


세 공 모두 2024년 서울 시리즈 개막전(샌디에이고 대 LA 다저스) 1~2회에 나왔다. 첫
번째 공은 `zone_dist`가 +0.10ft로 규칙상 존에 걸쳤는데 볼이 선언됐고, 세 번째 공은
거의 같은 거리(+0.11ft)에서 스트라이크가 선언됐다. 경계 공은 이렇게 위치가 비슷해도
판정이 엇갈린다. 이 분석이 알고 싶은 것은 이런 엇갈림이 볼카운트나 경기 상황, 선수에
따라 한쪽으로 기우는가이다.

주자가 없는 베이스는 `<NA>`로 표시된다.

### 시즌 간 차이 확인

앞에서 본 것처럼 경계 공 스트라이크 비율이 두 시즌 사이에 꽤 다르다. 두 시즌을
합쳐서 분석할 계획이라 그냥 넘어갈 수 없다. 두 시즌은 존과 투구 위치를 같은 방식으로
측정했지만, 심판이 같은 기준으로 판정했는지는 데이터를 봐야 안다. 우선 월별로 나눠서
차이가 언제부터 생겼는지 본다.

In [18]:
borderline = check[check["is_borderline"] == 1]
borderline.pivot_table(
    index=borderline["game_date"].dt.month.rename("월"),
    columns="game_year", values="is_strike", aggfunc="mean",
).round(3)

game_year,2024,2025
월,,
3,0.495,0.483
4,0.499,0.425
5,0.492,0.435
6,0.500,0.418
7,0.504,0.420
8,0.514,0.423
9,0.502,0.428


2024년은 시즌 내내 50% 안팎에서 거의 움직이지 않는다. 2025년도 3월(48.3%)까지는
비슷하다가 4월부터 42~43%로 내려가더니 시즌이 끝날 때까지 그대로다. 시즌 중에
서서히 바뀐 게 아니라 시즌 초에 한 번 뚝 떨어졌다.

이번에는 존의 어느 쪽에서 차이가 나는지 본다. 경계 공을 가장 가까운 경계가
좌우인지, 위쪽인지, 아래쪽인지로 나눈다. 좌우 경계는 홈플레이트 폭으로 정해져서 존
높이 기록(`sz_top`, `sz_bot`)과 상관이 없다. 좌우에서도 차이가 난다면 존 높이를 재는
방식이 바뀐 탓으로는 설명할 수 없다.

In [19]:
dx = (PLATE_HALF + BALL_RADIUS) - borderline["plate_x"].abs()
dz = np.minimum(
    (borderline["sz_top"] + BALL_RADIUS) - borderline["plate_z"],
    borderline["plate_z"] - (borderline["sz_bot"] - BALL_RADIUS),
)
mid_height = (borderline["sz_top"] + borderline["sz_bot"]) / 2
edge = np.where(dx < dz, "좌우", np.where(borderline["plate_z"] > mid_height, "상단", "하단"))

borderline.pivot_table(
    index=pd.Series(edge, index=borderline.index, name="가까운 경계"),
    columns="game_year", values="is_strike", aggfunc=["mean", "size"],
).round(3)

mean          size       
game_year   2024   2025   2024   2025
가까운 경계                               
상단         0.369  0.279  10181  10583
좌우         0.585  0.497  34321  34885
하단         0.402  0.369  15003  15080

위쪽, 아래쪽, 좌우 모두 2025년이 낮다. 좌우 경계에서도 58.5%에서 49.7%로
떨어졌으니, 존 높이 측정이 바뀌어서 생긴 차이는 아니다.

In [20]:
bins = np.arange(-0.3, 0.31, 0.1)
check.pivot_table(
    index=pd.cut(check["zone_dist"], bins).rename("zone_dist 구간 (ft)"),
    columns="game_year", values="is_strike", aggfunc="mean", observed=True,
).round(3)

game_year,2024,2025
zone_dist 구간 (ft),,
"(-0.3, -0.2]",0.053,0.031
"(-0.2, -0.1]",0.163,0.116
"(-0.1, 2.78e-17]",0.347,0.272
"(2.78e-17, 0.1]",0.670,0.594
"(0.1, 0.2]",0.857,0.830
"(0.2, 0.3]",0.937,0.929


존 바깥쪽(음수 구간)이든 존 안쪽 가장자리(0~0.1ft)든 2025년의 스트라이크 비율이
더 낮고, 경계에서 멀어질수록 두 시즌이 비슷해진다. 2025년 심판들의 판정 존이
전체적으로 조금 안쪽으로 좁아졌다고 볼 수 있다. 앞에서 본 프레이밍 리더보드의
`pct_tot`와도 방향이 같다.

두 시즌을 아무 처리 없이 합치면 이 차이가 상황이나 선수의 효과인 것처럼 섞여 들어갈
수 있다. 시즌마다 존이 조금씩 다른 건 좌타자와 우타자의 존이 다른 것과 마찬가지로
존이 어디에 있느냐의 문제이고, 이 분석이 찾으려는 판정의 쏠림과는 다르다. 그래서
이후 노트북에서는 모든 모델에 시즌(`game_year`)을 변수로 넣어 두 시즌의 평균적인
차이를 걸러 낸다. 그리고 주요 결과는 마지막에 시즌별로 따로 구해 다시 비교한다. 두
시즌의 효과가 비슷하게 나오면 존이 바뀌어도 쏠림은 그대로 남았다는 뜻이고,
2025년에 줄었다면 존이 좁아진 해에 쏠림도 함께 줄었다는 결과가 된다.

## 정리와 다음 단계

이 노트북에서 2024-2025 정규시즌 판정구 735,718개를 정리해 `data/clean.parquet`에
담았다. 원본은 `data/raw/`에, 포수 프레이밍 점수는 `data/framing_2024.csv`와
`data/framing_2025.csv`에 있다. 원본은 저장소에 올리지 않았지만, 이 노트북을
처음부터 실행하면 똑같이 다시 만들어진다.

다음 노트북부터는 이 파일 하나로 분석을 이어 간다. `01_baseline.ipynb`에서는 투구
위치와 타자 좌우, 시즌만으로 스트라이크 확률을 예측하는 기준 모델을 만든다. 공이
어디로 들어왔는지만 알 때 심판이 스트라이크를 부를 확률이다. 그 위에 볼카운트·아웃·
점수 차 같은 경기 상황과 투수·타자의 평판을 더한 모델들을 설계하고, 기준 모델과
나란히 놓고 비교한다. 같은 위치의 공이라도 상황이나 선수에 따라 판정이 달라진다면,
이런 정보를 더한 모델이 기준 모델보다 판정을 더 잘 맞힐 것이다. 그 차이가 우연이라고
보기 어려울 만큼 큰지 통계적으로 검정하는 것이 앞으로 할 일이다.